## Dependencies

In [ ]:
# Install dependencies
%pip install langchain langchain-core langchain-openai langchain-community langchain-text-splitters chromadb pypdf python-dotenv -q

# Verify installation
import sys
print(f"Python version: {sys.version}")

# Verify key packages are installed
try:
    import langchain
    import langchain_core
    print(f"LangChain version: {langchain.__version__}")
    print("All core packages installed successfully!")
except ImportError as e:
    print(f"⚠️  Error: {e}")
    print("Please ensure all packages are installed correctly.")


## Setup & Imports

In [ ]:
# Standard library imports
import os
from pathlib import Path
from dotenv import load_dotenv

# LangChain core imports
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# Modern LangChain agent imports (latest pattern from LangChain docs)
from langchain.agents import create_agent
from langchain.tools import tool

# Load environment variables - check current directory first, then parent
current_dir = Path.cwd()
env_path = current_dir / '.env'
if env_path.exists():
    load_dotenv(env_path)
    print(f"✅ Loading .env from: {env_path}")
else:
    # Try parent directory
    parent_env = current_dir.parent / '.env'
    if parent_env.exists():
        load_dotenv(parent_env)
        print(f"✅ Loading .env from: {parent_env}")
    else:
        # Fallback to default behavior (searches current and parent directories)
        load_dotenv()
        print("⚠️  No .env file found. Using default load_dotenv() search")

# Verify API key is set
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("⚠️  WARNING: OPENAI_API_KEY not found in environment variables!")
    print("Please create a .env file in this directory with your OpenAI API key:")
    print(f"   {current_dir / '.env'}")
    print("\nFormat: OPENAI_API_KEY=sk-...")
else:
    # Validate API key format (basic check)
    if not api_key.startswith('sk-'):
        print("⚠️  WARNING: API key format looks incorrect. Should start with 'sk-'")
    else:
        print("OpenAI API key loaded successfully!")
        # Test the API key by making a simple call
        try:
            test_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, max_tokens=5)
            test_llm.invoke("Hi")
            print("✅ API key validated successfully!")
        except Exception as e:
            print(f"❌ ERROR: API key validation failed!")
            print(f"   Error: {str(e)}")
            print("\n💡 Please check:")
            print("   1. Your API key is correct (get it from https://platform.openai.com/api-keys)")
            print("   2. You have sufficient credits in your OpenAI account")
            print("   3. The .env file is in the correct location")
            raise

print("\nAll imports successful!")
print("Using modern LangChain patterns: create_agent with @tool decorator")

✅ Loading .env from: /Users/kaliml/Cursor Projects/rag-qa-system/.env
✅ OpenAI API key loaded successfully!
✅ API key validated successfully!

✅ All imports successful!
📚 Using modern LangChain patterns: create_agent with @tool decorator


## Step 1: Load

- Use a LangChain document loader appropriate for your data type
- Print: number of documents loaded, sample of first document content

In [ ]:
#Load PDF
loader = PyPDFLoader("docs/CrossFit_Safety_And_Intensity_Are_Not_at_Odds_in_CrossFit.pdf")
documents = loader.load()      # Returns list of Document objects


print(f"Loaded {len(documents)} document(s)")
print(f"First document preview:")
print("-" * 60)
print(documents[0].page_content[:300] + "...")
print("-" * 60)
print(f"\n Document metadata: {documents[0].metadata}")


Ignoring wrong pointing object 100 0 (offset 0)
Ignoring wrong pointing object 101 0 (offset 0)


✅ Loaded 1 document(s)
📄 First document preview:
------------------------------------------------------------
COMMENTS ON SAFETY AND INTENSITY ARE NOT AT ODDS IN CROSSFIT
1 Comments Share Print LOG IN TO COMMENT
Back to 260315
Amedeo Alessio Cerea
March 15th, 2026 at 12:30 pm
Commented on: Safety and Intensity Are Not at Odds in CrossFit
Oh yes
Reply Share Print
ABOUT CROSSFIT
What Is
CrossFit?
Get Started
...
------------------------------------------------------------

📊 Document metadata: {'producer': 'macOS Version 26.3.1 (Build 25D2128) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20260316180226Z00'00'", 'moddate': "D:20260316180226Z00'00'", 'source': 'docs/CrossFit_Safety_And_Intensity_Are_Not_at_Odds_in_CrossFit.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}


## Step 2: Chunk

- Use RecursiveCharacterTextSplitter
- Print: total chunks created, character count of smallest and largest chunk
- Experiment: Try at least 2 different chunk_size values (e.g., 500 vs 1000) and note what changes

#### Tips 
- Start with a small dataset. 1-3 documents is plenty. Don't try to index 500 PDFs on your first attempt.
- If retrieval is bad, stop. Don't wire up the LLM hoping it'll fix bad retrieval. Debug chunking and embeddings first.
- Print intermediate outputs. Print your chunks. Print your retrieved docs. Print your prompt. Visibility = debugging speed.
- Document your failures. "I tried chunk_size=200 and it broke because..." is more valuable than a clean notebook that hides the messy parts.

In [ ]:
# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # Size of each chunk
    chunk_overlap=100,   # Overlap between chunks (important!)
    separators=["\n\n", "\n", ". ", " ", ""]  # Try to split at these boundaries
)

# Split documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"Split document into {len(chunks)} chunks")
print(f"\nChunk statistics:")
print(f"  Average chunk size: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} characters")
print(f"  Min chunk size: {min(len(c.page_content) for c in chunks)} characters")
print(f"  Max chunk size: {max(len(c.page_content) for c in chunks)} characters")

print(f"\nFirst chunk preview:")
print("-" * 60)
print(chunks[0].page_content)
print("-" * 60)

✅ Split document into 14 chunks

📊 Chunk statistics:
  Average chunk size: 748 characters
  Min chunk size: 567 characters
  Max chunk size: 797 characters

📄 First chunk preview:
------------------------------------------------------------
COMMENTS ON SAFETY AND INTENSITY ARE NOT AT ODDS IN CROSSFIT
1 Comments Share Print LOG IN TO COMMENT
Back to 260315
Amedeo Alessio Cerea
March 15th, 2026 at 12:30 pm
Commented on: Safety and Intensity Are Not at Odds in CrossFit
Oh yes
Reply Share Print
ABOUT CROSSFIT
What Is
CrossFit?
Get Started
Workouts
Movements
FAQ
Help Center
Careers
EDUCATION
Courses Near
You
Certificate
Courses
Certifications
Explore Courses
Promotional
Terms and
Conditions
AFFILIATES
Open a CrossFit
Gym
Field Leaders
Global Mentor
Program
Affiliate Toolkit
COMMUNITY
Find a Trainer
Scholarship
Program
Foundation
CrossFit
Medical Society
THE CROSSFIT GAMES
About the
Games
Leaderboard
Schedule
Workouts
FIND A GYM TODAY!
Start your fitness journey today and get
healthy.
FIND A

## Step 3: Embed + Store

- Embed chunks using an embedding model of your choice
- Store in ChromaDB (or another vector DB if you're feeling adventurous)

In [ ]:
# Initialize embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Clear the ChromaDB (Optional)
import shutil

# Clear existing vector database so we start fresh every run
db_path = "./rag_vector_db"
if os.path.exists(db_path):
    shutil.rmtree(db_path)
    print("Cleared existing vector database")

# Now create fresh
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=db_path
)
print("Created fresh vector database!")
print(f"Stored {len(chunks)} document chunks")

# Create vector store from chunks
# This will:
# 1. Generate embeddings for all chunks
# 2. Store them in ChromaDB
# 3. Persist to disk for later use

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./rag_vector_db"
)

print("Created vector database!")
print(f"Database saved to: ./rag_vector_db")
print(f"Stored {len(chunks)} document chunks")



✅ Created vector database!
📁 Database saved to: ./rag_vector_db
📊 Stored 14 document chunks


# Step 4: Test Retrieval (before wiring up the LLM!)

- Run 3 test queries using similarity_search
- For each query, print the top 3 retrieved chunks
- Annotate: Are the retrieved chunks actually relevant? Yes/No and why?

In [ ]:
#Retrieval Test Question setup
questions = [
    "What is the athlete responsible for?",
    "What is consistency?",
    "What is the coach responsible for?",
]

for question in questions:
    results = vectorstore.similarity_search(question, k=3)

    print(f"\nTEST QUERY: '{question}'")
    print(f"\nRETRIEVED {len(results)} RELEVANT CHUNKS:")
    print("-" * 60)
    for i, doc in enumerate(results, 1):
        print(f"\n{i}. {doc.page_content[:200]}...")
        print(f"   METADATA: {doc.metadata}")
        print("-" * 15)



🔍 Test query: 'What is the athlete responsible for?'

📄 Retrieved 3 relevant chunks:
------------------------------------------------------------

1. assessing your athlete’s body language and struggles during the general and
specific warm-up. 
The coach is also responsible for effectively implementing quality warm-ups
and cool-downs. Warm-ups prep...
   Metadata: {'moddate': "D:20260316180226Z00'00'", 'page_label': '1', 'producer': 'macOS Version 26.3.1 (Build 25D2128) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20260316180226Z00'00'", 'total_pages': 1, 'source': 'docs/CrossFit_Safety_And_Intensity_Are_Not_at_Odds_in_CrossFit.pdf', 'page': 0}
---------------

2. athletes’ data and attendance.
As athletes advance, the coach will continue to refine more subtle faults to help
them improve their movement efficiency and increase intensity (power). The
coach is als...
   Metadata: {'creator': 'PyPDF', 'source': 'docs/CrossFit_Safety_And_Intensity_Are_Not_at_Odds_in_CrossFit.

## Step 5: Build the RAG Chain

- Wire up RetrievalQA with a custom prompt template
- Run the same 3 queries through the full chain
- Print the generated answers

In [ ]:
# Initialize LLM
model = ChatOpenAI(
    model="gpt-4o-mini",  # Fast and cost-effective
    temperature=0  # Deterministic responses
)

# Create a retrieval tool using the modern @tool decorator pattern
# This follows the latest LangChain documentation pattern
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information from the document store to help answer a query."""
    retrieved_docs = vectorstore.similarity_search(query, k=3)
    serialized = "\n\n".join(
        (f"SOURCE: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

# Create the RAG agent using create_agent (modern pattern)
tools = [retrieve_context]
system_prompt = (
    "You have access to a tool that retrieves context from documents. "
    "Use the tool to help answer user queries.  Only provide information from the documents - do not embellish. Always cite your sources."
)

rag_agent = create_agent(model, tools, system_prompt=system_prompt)

print("\nAGENT CONFIGURATION:")
print(f"  LLM: {model.model_name}")
print(f"  RETRIEVAL: Top 3 chunks")
print(f"  PATTERN: Agent with @tool decorator (latest LangChain docs)")

✅ RAG agent created using modern LangChain pattern!

📋 Agent configuration:
  LLM: gpt-4o-mini
  Retrieval: Top 3 chunks
  Pattern: Agent with @tool decorator (latest LangChain docs)


In [ ]:
#Use the questions defined previously

for question in questions:
    print(f"\n{'='*70}")
    print(f"❓ Question: {question}")
    print(f"{'='*70}\n")

    for event in rag_agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="values",
    ):
        last_message = event["messages"][-1]

        # Only print the final answer (skip tool calls and raw tool results)
        has_tool_calls = hasattr(last_message, "tool_calls") and last_message.tool_calls
        has_content = hasattr(last_message, "content") and last_message.content

        if has_content and not has_tool_calls:
            # Skip the echoed question (it's just a HumanMessage)
            if last_message.__class__.__name__ != "HumanMessage":
                print(f"💬 Answer:\n{last_message.content}\n")

    print("-" * 70)


❓ Question: What is the athlete responsible for?

💬 Answer:
Source: {'moddate': "D:20260316180226Z00'00'", 'producer': 'macOS Version 26.3.1 (Build 25D2128) Quartz PDFContext', 'creator': 'PyPDF', 'page': 0, 'page_label': '1', 'total_pages': 1, 'source': 'docs/CrossFit_Safety_And_Intensity_Are_Not_at_Odds_in_CrossFit.pdf', 'creationdate': "D:20260316180226Z00'00'"}
Content: assessing your athlete’s body language and struggles during the general and
specific warm-up. 
The coach is also responsible for effectively implementing quality warm-ups
and cool-downs. Warm-ups prepare the body for upcoming demands, allow
coaches time for skill refinement and load management, and lay the
groundwork for effective scaling. Effective cool-downs also allow athletes to
recover more quickly and be prepared for the demands of the days ahead. 
The Athlete
The athlete’s primary role is to check their ego and listen to their coach’s
guidance on scaling, movement refinement, and quality programming. The
ath

## Step 6: Evaluate (the part most people skip)

- Create a mini eval set: 5 questions where you know the correct answer
- Run all 5 through your pipeline
- For each, record:

    ✅ or ❌ Did it retrieve the right chunks?
    ✅ or ❌ Is the answer grounded in the context (not hallucinated)?
    ✅ or ❌ Is the answer actually correct?
- Calculate your accuracy: X/5 retrieval, X/5 faithfulness, X/5 correctness

In [ ]:
eval_questions = [
    "How do I achieve sustainable intensity?",
    "Why are warm-up routines important?",
    "What needs to happen before increasing load?",
    "Why would an athlete's intensity need to change?",
    "Who is responsible for managing threshold training?"
]

# Expected answers (for manual scoring)
expected_answers = [
    "Safety is the Foundation for Sustainable Intensity",
    "Warm-ups prepare the body for upcoming demands, allow coaches time for skill refinement and load management, and lay the groundwork for effective scaling",
    "Coaches guide athletes on a path to develop correct technique. After mechanics and consistency are established, we can then start to focus on increasing intensity",
    "An athlete's relative intensity may change frequently. For example, an athlete might have a poor night's sleep or be in a period of high stress.",
    "The coach is responsible for managing this for every workout, for every athlete."
]

results = []

for i, question in enumerate(eval_questions):
    print(f"\n{'='*70}")
    print(f"❓ Question {i+1}: {question}")
    print(f"{'='*70}\n")

    final_answer = None

    for event in rag_agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="values",
    ):
        last_message = event["messages"][-1]
        has_tool_calls = hasattr(last_message, "tool_calls") and last_message.tool_calls
        has_content = hasattr(last_message, "content") and last_message.content
        is_human = last_message.__class__.__name__ == "HumanMessage"

        if has_content and not has_tool_calls and not is_human:
            final_answer = last_message.content

    print(f"💬 Answer:\n{final_answer}\n")
    print(f"📋 Expected:\n{expected_answers[i]}\n")

    # Manual scoring — fill these in as you review
    results.append({
        "question": question,
        "answer": final_answer,
        "retrieved_right_chunks": None,   # ✅ True / ❌ False
        "grounded_not_hallucinated": None,
        "answer_is_correct": None
    })

    print("-" * 70)

# Summary (fill in scores above first, then re-run this block)
print(f"\n{'='*70}")
print("📊 EVAL SUMMARY")
print(f"{'='*70}")
scored = [r for r in results if r["retrieved_right_chunks"] is not None]
if scored:
    retrieval  = sum(1 for r in scored if r["retrieved_right_chunks"]) 
    grounded   = sum(1 for r in scored if r["grounded_not_hallucinated"])
    correct    = sum(1 for r in scored if r["answer_is_correct"])
    n = len(scored)
    print(f"  Retrieval:    {retrieval}/{n}")
    print(f"  Faithfulness: {grounded}/{n}")
    print(f"  Correctness:  {correct}/{n}")
else:
    print("  Fill in scores in the results list above, then re-run this cell.")

💬 Answer:
What is the coach responsible for?

🔧 Tool Calls:
  - retrieve_context: {'query': 'coach responsibilities'}

💬 Answer:
Source: {'total_pages': 1, 'creationdate': "D:20260316180226Z00'00'", 'source': 'docs/CrossFit_Safety_And_Intensity_Are_Not_at_Odds_in_CrossFit.pdf', 'page': 0, 'page_label': '1', 'producer': 'macOS Version 26.3.1 (Build 25D2128) Quartz PDFContext', 'creator': 'PyPDF', 'moddate': "D:20260316180226Z00'00'"}
Content: lifted and higher-level skills to be developed. 
Managing Safety and Intensity at an Affiliate
Managing safety and intensity at an affiliate should be a collaborative effort with
the coach and the athlete. However, the large bulk of this demand is placed on
the coach. 
The Coach
The coaching staff is the most integral part of providing athletes with an
experience that keeps them safe while delivering results. Coaches guide
athletes on a path to develop correct technique before significantly increasing
loads or speed. In other words, they guide them